# Ноутбук 10 — Эксперимент 7: Статистическая значимость различий моделей

**Назначение.** Парная проверка статистической значимости различий между
моделями, использованными в работе, помимо сравнения средних значений
метрик.

**Тесты:**
1. **Wilcoxon signed-rank test** (двусторонний, `zero_method='wilcox'`) для
   парных сравнений по 32 LOSO-фолдам — оконный уровень (exp01, exp02,
   exp04 ablations, exp06 deep).
2. **McNemar exact test** для блочного уровня (exp03, 32 субъекта,
   бинарные предсказания).
3. **Holm–Bonferroni correction** для семи ключевых сравнений оконного
   уровня (множественные сравнения).

**Главный вывод:** ключевые сравнения (RF vs XGBoost, RF vs LightGBM,
XGBoost vs LR на блоке, eye vs combined ≡, physio < combined) проходят
тест значимости; различие RF vs LR на оконном уровне после поправки
Холма теряет значимость; LSTM/BiLSTM статистически эквивалентны RF.


In [1]:
import sys
from pathlib import Path

NB_ROOT = Path.cwd()
if str(NB_ROOT) not in sys.path:
    sys.path.insert(0, str(NB_ROOT))

import warnings
warnings.filterwarnings("ignore")

import numpy as np
import pandas as pd
from scipy.stats import wilcoxon
from statsmodels.stats.contingency_tables import mcnemar

from modules import config

R = config.RESULTS_DIR
print(f"Results dir: {R}")


Results dir: D:\Programming\Python\Diplom\source\results


## 1. Wilcoxon: оконный уровень exp01 (RF vs LR baseline)

In [2]:
exp01 = pd.read_csv(R / "exp01_metrics.csv")
loso = exp01[exp01.strategy == "LOSO"].sort_values("fold")
piv_f1 = loso.pivot(index="fold", columns="model", values="f1_macro")
piv_acc = loso.pivot(index="fold", columns="model", values="accuracy")

stat_f1, p_f1 = wilcoxon(piv_f1["RF"], piv_f1["LogReg"], alternative="two-sided", zero_method="wilcox")
stat_acc, p_acc = wilcoxon(piv_acc["RF"], piv_acc["LogReg"], alternative="two-sided", zero_method="wilcox")

print(f"RF vs LR (F1):  mean {piv_f1['RF'].mean():.4f} vs {piv_f1['LogReg'].mean():.4f}  W={stat_f1:.1f}  p={p_f1:.4f}")
print(f"RF vs LR (acc): mean {piv_acc['RF'].mean():.4f} vs {piv_acc['LogReg'].mean():.4f}  W={stat_acc:.1f}  p={p_acc:.4f}")


RF vs LR (F1):  mean 0.5748 vs 0.5450  W=143.0  p=0.0228
RF vs LR (acc): mean 0.6322 vs 0.5961  W=78.0  p=0.0009


## 2. Wilcoxon: оконный уровень exp02 (4 модели, persubj-таргет)

In [3]:
exp02 = pd.read_csv(R / "exp02_metrics.csv")
loso = exp02[exp02.strategy == "LOSO"].sort_values("fold")
piv_f1 = loso.pivot(index="fold", columns="model", values="f1_macro")
print("model means F1-macro:")
print(piv_f1.mean().round(4).to_string())


model means F1-macro:
model
LightGBM              0.6018
LogisticRegression    0.6080
RandomForest          0.6189
XGBoost               0.6020


## 3. Перезапуск LSTM/BiLSTM с сохранением per-fold скоров

In [4]:
import time
import torch

from modules.experiments import (
    NON_FEATURE_COLS, add_per_subject_target, per_subject_zscore,
)
from modules.deep import build_sequences, train_fold, K_DEFAULT

DEEP_OUT = R / "exp07_deep_perfold.csv"

if DEEP_OUT.exists():
    print(f"{DEEP_OUT.name} уже существует — пропускаем перезапуск")
    deep_df = pd.read_csv(DEEP_OUT)
else:
    df = pd.read_csv(R / "feature_table.csv")
    df = add_per_subject_target(df)
    feat = [c for c in df.columns if c not in NON_FEATURE_COLS and c != "arousal_class_persubj"]
    y = df["arousal_class_persubj"].to_numpy(dtype=np.int64)
    X = per_subject_zscore(df, feat).astype(np.float32)
    X = np.nan_to_num(X, nan=0.0, posinf=0.0, neginf=0.0)
    seq_X, seq_y, seq_pid = build_sequences(df, X, y, k=K_DEFAULT)
    print(f"sequences: {seq_X.shape}  class balance: {np.bincount(seq_y).tolist()}")

    rows = []
    for arch_name, bidir in [("LSTM", False), ("BiLSTM", True)]:
        print(f"\n=== {arch_name} LOSO ===")
        t0 = time.time()
        for fi, pid in enumerate(np.unique(seq_pid)):
            te = seq_pid == pid
            tr = ~te
            if len(np.unique(seq_y[tr])) < 2 or te.sum() == 0:
                continue
            mt = train_fold(seq_X[tr], seq_y[tr], seq_X[te], seq_y[te],
                            bidirectional=bidir, n_features=seq_X.shape[2])
            rows.append({"model": arch_name, "fold": fi, "participant": str(pid), **mt})
            if (fi + 1) % 8 == 0:
                print(f"  fold {fi+1}/32 {pid} acc={mt['accuracy']:.4f} f1={mt['f1_macro']:.4f}  ({time.time()-t0:.0f}s)")
        print(f"  total {time.time()-t0:.0f}s")

    deep_df = pd.DataFrame(rows)
    deep_df.to_csv(DEEP_OUT, index=False)
    print(f"\nsaved {DEEP_OUT.name} {deep_df.shape}")

print(deep_df.groupby("model")[["accuracy", "f1_macro", "auc_roc"]].mean().round(4))


sequences: (12800, 10, 143)  class balance: [7104, 5696]

=== LSTM LOSO ===


  fold 8/32 P16 acc=0.5925 f1=0.5713  (23s)


  fold 16/32 P23 acc=0.6900 f1=0.6855  (48s)


  fold 24/32 P30 acc=0.6100 f1=0.5623  (72s)


  fold 32/32 P9 acc=0.6500 f1=0.6305  (96s)
  total 96s

=== BiLSTM LOSO ===


  fold 8/32 P16 acc=0.5750 f1=0.5298  (40s)


  fold 16/32 P23 acc=0.6625 f1=0.6545  (81s)


  fold 24/32 P30 acc=0.6425 f1=0.6010  (121s)


  fold 32/32 P9 acc=0.6100 f1=0.6057  (161s)
  total 161s

saved exp07_deep_perfold.csv (64, 8)
        accuracy  f1_macro  auc_roc
model                              
BiLSTM    0.6123    0.6003   0.6516
LSTM      0.6115    0.6007   0.6563


## 4. Перезапуск abляций по модальностям (eye/physio/combined) с per-fold

In [5]:
from modules.experiments import (
    classify_feature, select_columns, make_random_forest_exp02,
)
from modules.validation import (
    compute_classification_metrics, subject_independent_splits,
)

ABL_OUT = R / "exp07_ablation_perfold.csv"

if ABL_OUT.exists():
    print(f"{ABL_OUT.name} уже существует — пропускаем")
    abl_df = pd.read_csv(ABL_OUT)
else:
    df = pd.read_csv(R / "feature_table.csv")
    df = add_per_subject_target(df)
    ALL = [c for c in df.columns if c not in NON_FEATURE_COLS and c != "arousal_class_persubj"]
    y = df["arousal_class_persubj"].to_numpy(dtype=int)
    splits = list(subject_independent_splits(df))

    rows = []
    for variant, mod in [("eye", "eye"), ("physio", "physio"), ("combined", None)]:
        cols = select_columns(ALL, modality=mod) if mod else ALL
        X = per_subject_zscore(df, cols)
        print(f"ablation modality={variant} n_features={len(cols)}")
        t0 = time.time()
        for fi, (tr, te) in enumerate(splits):
            m = make_random_forest_exp02()
            m.fit(X[tr], y[tr])
            mt = compute_classification_metrics(y[te], m.predict(X[te]), m.predict_proba(X[te]))
            rows.append({"variant": variant, "fold": fi, "n_features": len(cols), **mt})
        print(f"  done in {time.time()-t0:.1f}s")

    abl_df = pd.DataFrame(rows)
    abl_df.to_csv(ABL_OUT, index=False)
    print(f"saved {ABL_OUT.name} {abl_df.shape}")

print(abl_df.groupby("variant")[["accuracy", "f1_macro"]].mean().round(4))


ablation modality=eye n_features=90


  done in 45.8s
ablation modality=physio n_features=53


  done in 41.6s


ablation modality=combined n_features=143


  done in 56.4s
saved exp07_ablation_perfold.csv (96, 8)
          accuracy  f1_macro
variant                     
combined    0.6526    0.6189
eye         0.6535    0.6233
physio      0.5289    0.4640


## 5. Wilcoxon на оконном уровне: 7 ключевых сравнений + Holm-Bonferroni

In [6]:
# Собираем pivot F1 для exp02 + добавляем LSTM/BiLSTM из deep_df
exp02_loso = exp02[exp02.strategy == "LOSO"].sort_values("fold")
piv = exp02_loso.pivot(index="fold", columns="model", values="f1_macro").sort_index()

lstm_f1 = deep_df[deep_df.model == "LSTM"].sort_values("fold").f1_macro.to_numpy()
bilstm_f1 = deep_df[deep_df.model == "BiLSTM"].sort_values("fold").f1_macro.to_numpy()

def get_scores(name):
    if name in piv.columns:
        return piv[name].to_numpy()
    if name == "LSTM":
        return lstm_f1
    if name == "BiLSTM":
        return bilstm_f1
    raise KeyError(name)

PAIRS = [
    ("RandomForest", "LogisticRegression"),
    ("RandomForest", "XGBoost"),
    ("RandomForest", "LightGBM"),
    ("XGBoost", "LightGBM"),
    ("RandomForest", "LSTM"),
    ("RandomForest", "BiLSTM"),
    ("LSTM", "BiLSTM"),
]

raw = []
for a, b in PAIRS:
    A, B = get_scores(a), get_scores(b)
    stat, p = wilcoxon(A, B, alternative="two-sided", zero_method="wilcox")
    raw.append({"A": a, "B": b, "mean_A": float(A.mean()), "mean_B": float(B.mean()),
                "W": float(stat), "p_raw": float(p)})

# Holm step-down
m = len(raw)
order = sorted(range(m), key=lambda i: raw[i]["p_raw"])
for rank, i in enumerate(order):
    raw[i]["p_holm"] = min(1.0, raw[i]["p_raw"] * (m - rank))
    raw[i]["significant_holm"] = raw[i]["p_holm"] < 0.05

window_df = pd.DataFrame(raw)
window_df.to_csv(R / "exp07_wilcoxon_window_holm.csv", index=False)
print(window_df.to_string(index=False))


           A                  B   mean_A   mean_B     W    p_raw   p_holm  significant_holm
RandomForest LogisticRegression 0.618928 0.608029 151.0 0.034120 0.170600             False
RandomForest            XGBoost 0.618928 0.602033 113.0 0.003866 0.023194              True
RandomForest           LightGBM 0.618928 0.601842 112.0 0.003615 0.025307              True
     XGBoost           LightGBM 0.602033 0.601842 213.0 0.492789 0.985579             False
RandomForest               LSTM 0.618928 0.600675 208.0 0.303787 1.000000             False
RandomForest             BiLSTM 0.618928 0.600291 213.0 0.349837 1.000000             False
        LSTM             BiLSTM 0.600675 0.600291 240.0 0.664522 0.664522             False


## 6. McNemar: блочный уровень (exp03, 32 субъекта)

In [7]:
preds = pd.read_csv(R / "exp03_predictions.csv")
y_true = preds.y_true.to_numpy()

MC_PAIRS = [
    ("XGBoost", "LogReg"),
    ("XGBoost", "RF"),
    ("RF", "LogReg"),
]

mc_rows = []
for a, b in MC_PAIRS:
    pa = preds[f"{a}_pred"].to_numpy()
    pb = preds[f"{b}_pred"].to_numpy()
    ca = (pa == y_true).astype(int)
    cb = (pb == y_true).astype(int)
    n11 = int(((ca == 1) & (cb == 1)).sum())
    n10 = int(((ca == 1) & (cb == 0)).sum())  # b
    n01 = int(((ca == 0) & (cb == 1)).sum())  # c
    n00 = int(((ca == 0) & (cb == 0)).sum())
    res = mcnemar([[n11, n10], [n01, n00]], exact=True)
    mc_rows.append({"A": a, "B": b, "A_acc": float(ca.mean()), "B_acc": float(cb.mean()),
                    "b_only_A": n10, "c_only_B": n01,
                    "stat": float(res.statistic), "p": float(res.pvalue)})

mc_df = pd.DataFrame(mc_rows)
mc_df.to_csv(R / "exp07_mcnemar.csv", index=False)
print(mc_df.to_string(index=False))


      A      B   A_acc   B_acc  b_only_A  c_only_B  stat        p
XGBoost LogReg 0.71875 0.43750        11         2   2.0 0.022461
XGBoost     RF 0.71875 0.53125         8         2   2.0 0.109375
     RF LogReg 0.53125 0.43750         6         3   3.0 0.507812


## 7. Wilcoxon на абляциях по модальностям

In [8]:
abl_piv = abl_df.pivot(index="fold", columns="variant", values="f1_macro")
ABL_PAIRS = [("combined", "eye"), ("combined", "physio"), ("eye", "physio")]
abl_rows = []
for a, b in ABL_PAIRS:
    stat, p = wilcoxon(abl_piv[a], abl_piv[b], alternative="two-sided", zero_method="wilcox")
    abl_rows.append({"A": a, "B": b,
                     "mean_A": float(abl_piv[a].mean()), "mean_B": float(abl_piv[b].mean()),
                     "W": float(stat), "p": float(p)})

abl_w_df = pd.DataFrame(abl_rows)
abl_w_df.to_csv(R / "exp07_wilcoxon_ablation.csv", index=False)
print(abl_w_df.to_string(index=False))


       A      B   mean_A   mean_B     W            p
combined    eye 0.618928 0.623263 201.0 2.462135e-01
combined physio 0.618928 0.464002   3.0 2.328306e-09
     eye physio 0.623263 0.464002   1.0 9.313226e-10


## 8. Итог

Сохранены файлы:
- `exp07_wilcoxon_window_holm.csv` — 7 ключевых сравнений на оконном уровне (Wilcoxon + Holm).
- `exp07_mcnemar.csv` — 3 пары моделей на блочном уровне (McNemar exact).
- `exp07_wilcoxon_ablation.csv` — 3 сравнения модальностей.
- `exp07_ablation_perfold.csv` — per-fold скоры абляций (используется выше).
- `exp07_deep_perfold.csv` — per-fold скоры LSTM/BiLSTM (используется выше).